In [1]:
from pathlib import Path
import geopandas as gpd
import polars as pl

### Load files
1. List all gold parquet files that are expected from the pipeline

In [2]:
gold_dir = Path('../data/gold')
adm1_path = Path('../data/boundaries/geoBoundaries-IDN-ADM1-provinces.geojson')
adm2_path = Path('../data/boundaries/geoBoundaries-IDN-ADM2-districts.geojson')

required_gold_files = [
    'building_count_country.parquet',
    'building_count_quadkey.parquet',
    'building_counts_by_province.parquet',
    'building_counts_by_district.parquet',
    'building_counts_by_province_unmatched.parquet',
    'building_counts_by_district_unmatched.parquet',
    'building_counts_by_province_unmatched_nearest_adm1.parquet',
    'building_counts_by_district_unmatched_nearest_adm2.parquet',
]

### Check required gold files
1. Compare the expected gold file list with what exists in data/gold
2. Print a success message if all files exist
3. Otherwise print the missing file names

In [3]:
missing_gold_files = [name for name in required_gold_files if not (gold_dir / name).exists()]

if not missing_gold_files:
    print('Gold files OK')
else:
    print('Missing gold files:')
    for name in missing_gold_files:
        print(name)

Gold files OK


### Load gold outputs
1. Read the country, quadkey, province, and district count parquet files
2. Read unmatched and recovered province/district parquet files
3. These tables are used for all reconciliation checks below

In [4]:
country_counts = pl.read_parquet(gold_dir / 'building_count_country.parquet')
quadkey_counts = pl.read_parquet(gold_dir / 'building_count_quadkey.parquet')
province_counts = pl.read_parquet(gold_dir / 'building_counts_by_province.parquet')
district_counts = pl.read_parquet(gold_dir / 'building_counts_by_district.parquet')

province_unmatched = gpd.read_parquet(gold_dir / 'building_counts_by_province_unmatched.parquet')
district_unmatched = gpd.read_parquet(gold_dir / 'building_counts_by_district_unmatched.parquet')
province_recovered = gpd.read_parquet(gold_dir / 'building_counts_by_province_unmatched_nearest_adm1.parquet')
district_recovered = gpd.read_parquet(gold_dir / 'building_counts_by_district_unmatched_nearest_adm2.parquet')

### Reconcile total building counts
1. Count totals from country, quadkey, province, and district outputs
2. Compare the totals across each aggregation level
3. Print mismatches if any total does not agree

In [5]:
country_total = int(country_counts['building_count'][0])
quadkey_total = int(quadkey_counts['building_count'].sum())
province_total = int(province_counts['building_count'].sum())
district_total = int(district_counts['building_count'].sum())

all_ok = True

if country_total != quadkey_total:
    print(f'country/quadkey mismatch: country={country_total} quadkey={quadkey_total}')
    all_ok = False

if country_total != province_total:
    print(f'country/province mismatch: country={country_total} province={province_total}')
    all_ok = False

if country_total != district_total:
    print(f'country/district mismatch: country={country_total} district={district_total}')
    all_ok = False

if all_ok:
    print('Total reconciliation OK')


Total reconciliation OK


### Check unmatched and recovered row counts
1. Compare the number of unmatched rows with the number of recovered rows
2. Run the check for both province and district outputs
3. Print a confirmation only if both row counts match

In [6]:
all_ok = True

if len(province_unmatched) != len(province_recovered):
    print(
        'province unmatched/recovered row mismatch: ' +
        f'unmatched={len(province_unmatched)} recovered={len(province_recovered)}'
    )
    all_ok = False

if len(district_unmatched) != len(district_recovered):
    print(
        'district unmatched/recovered row mismatch: ' +
        f'unmatched={len(district_unmatched)} recovered={len(district_recovered)}'
    )
    all_ok = False

if all_ok:
    print('Unmatched and recovered row counts OK')

Unmatched and recovered row counts OK


### Check recovered silver_record_id uniqueness
1. Count duplicate silver_record_id values in recovered province rows
2. Count duplicate silver_record_id values in recovered district rows
3. Print a confirmation only if all recovered IDs are unique

In [7]:
all_ok = True

province_duplicate_ids = int(province_recovered['silver_record_id'].duplicated().sum())
district_duplicate_ids = int(district_recovered['silver_record_id'].duplicated().sum())

if province_duplicate_ids > 0:
    print(f'province recovered assignments contain {province_duplicate_ids} duplicate silver_record_id values')
    all_ok = False

if district_duplicate_ids > 0:
    print(f'district recovered assignments contain {district_duplicate_ids} duplicate silver_record_id values')
    all_ok = False

if all_ok:
    print('Recovered silver_record_id uniqueness OK')

Recovered silver_record_id uniqueness OK


### Load valid province and district keys
1. Read ADM1 and ADM2 boundary files
2. Rename boundary columns to the same names used in gold outputs
3. Keep only the boundary key columns needed for validation

In [8]:
provinces = (
    gpd.read_file(adm1_path)[['shapeName', 'shapeISO']]
    .rename(columns={'shapeName': 'province_name', 'shapeISO': 'province_code'})
)
districts = (
    gpd.read_file(adm2_path)[['shapeName', 'shapeID']]
    .rename(columns={'shapeName': 'district_name', 'shapeID': 'district_code'})
)

### Validate final boundary keys and counts
1. Check province and district outputs for null boundary keys
2. Check for duplicate key rows and negative building_count values
3. Compare observed output keys against valid boundary keys from the shapefiles

In [9]:
all_ok = True

for label, df, name_columns, valid_df in [
    ('province', province_counts, ['province_name', 'province_code'], provinces),
    ('district', district_counts, ['district_name', 'district_code'], districts),
]:
    null_rows = df.filter(
        pl.any_horizontal([pl.col(column).is_null() for column in name_columns])
    ).height
    if null_rows > 0:
        print(f'{label} output contains {null_rows} rows with null boundary keys')
        all_ok = False

    duplicate_rows = df.group_by(name_columns).len().filter(pl.col('len') > 1).height
    if duplicate_rows > 0:
        print(f'{label} output contains {duplicate_rows} duplicate boundary key rows')
        all_ok = False

    negative_rows = df.filter(pl.col('building_count') < 0).height
    if negative_rows > 0:
        print(f'{label} output contains {negative_rows} negative building_count rows')
        all_ok = False

    observed_keys = {tuple(row) for row in df.select(name_columns).iter_rows()}
    valid_keys = {tuple(row) for row in valid_df[name_columns].drop_duplicates().itertuples(index=False, name=None)}
    unexpected_keys = sorted(observed_keys - valid_keys)
    if unexpected_keys:
        print(f'{label} output contains unexpected boundary keys: {unexpected_keys[:5]}')
        all_ok = False

if all_ok:
    print('Final output boundary keys OK')

Final output boundary keys OK
